# qlang · live expression explorer

**Run the setup cell once, then run the single “Launch / refresh” cell.** That cell loads all parser, graph, and widget code from the adjacent `live_ast_widgets.py` file every time. You no longer need to run definitions in several cells or restart the kernel after code changes.

The live view has an expression field, an AST graph, and an editable **Variables** table beside it (below it on narrow screens). This demo performs no quantum simulation.


In [2]:
# Install only missing packages into the selected notebook kernel.
# Keep GitHub sources; do not reinstall widget infrastructure on every Run All.
from importlib.metadata import version, PackageNotFoundError

requirements = {
    "qlang": "qlang @ git+https://github.com/asherhe/qlang.git@main",
    "ipywidgets": "ipywidgets @ git+https://github.com/jupyter-widgets/ipywidgets.git@8.1.7#subdirectory=python/ipywidgets",
}
missing = []
for package, requirement in requirements.items():
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        missing.append(requirement)
if missing:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    raise RuntimeError(
        "Installation finished. RESTART THE KERNEL, then Run All again. "
        "VS Code discovers widget support when the kernel connects; "
        "continuing in this first-install session can produce blank output."
    )
print("Dependencies ready. Continue below.")


qlang: 0.1.0
ipywidgets: 8.1.9
Dependencies ready. Continue below.


**First installation requires a kernel restart.** The setup cell stops after installing missing dependencies. Click **Restart**, then **Run All** again; installed packages are reused. If a previous session showed a blank widget, close/reopen this notebook after restarting. If it remains blank, use **Developer: Reload Window** in VS Code's command palette.

A plain HTML graph preview below works independently of the widget renderer. The **Test widget connection** button confirms that browser events reach Python. If the preview works but controls do not appear, the problem is the frontend widget renderer, not the AST. Check that the Microsoft Jupyter extension is enabled, and consult [VS Code's widget support guidance](https://github.com/microsoft/vscode-jupyter/wiki/IPyWidget-Support-in-VS-Code-Python).

**Inputs:** the variable table defines the available names. It starts with unsigned integers `x` and `y`, plus Boolean bits `p` and `q`. Use `&`, `|`, `^`, `~` and parenthesized comparisons. Calls, attributes, indexing, and Python `and/or/not` are excluded. This demo resets qlang's global registry; keep unrelated problems in another kernel.


## Launch / refresh
Run this cell whenever you reopen the notebook or want to pick up code changes. Look for **Live AST v2** in its output. If VS Code has an older unsaved notebook open, preserve that copy and reopen this file from disk.


In [3]:
# Launch / refresh — this is the only cell to rerun while using the demo.
import importlib.util
from pathlib import Path

print("Loading the current live AST demo from disk…")
roots = [Path.cwd(), *Path.cwd().parents]
candidates = [candidate for root in roots
              for candidate in (root / "live_ast_widgets.py", root / "src" / "qlang" / "live_ast_widgets.py")]
helper = next((candidate for candidate in candidates if candidate.is_file()), None)
if helper is None:
    raise FileNotFoundError("Keep live_ast_widgets.py beside this notebook, and use that folder or the repository as the kernel working directory.")
# Compile the current file directly; avoid cached imports or stale notebook definitions.
spec = importlib.util.spec_from_file_location("qlang_live_ast_demo", helper)
live_ast_demo = importlib.util.module_from_spec(spec)
exec(compile(helper.read_text(encoding="utf-8"), str(helper), "exec"), live_ast_demo.__dict__)
print("Loaded:", helper)
previous_explorer = globals().get("explorer")
explorer = live_ast_demo.show_explorer()
if previous_explorer is not None:
    previous_explorer.close()


Loading the current live AST demo from disk…
Loaded: C:\Users\asher\projects\qlang\examples\live_ast_widgets.py
Live AST v2: editable variable table and routed edges. Controls appear below.


## A one-minute demo
1. Start with `(x + y == 7) & (x > y)` and point out the comparison and arithmetic branches.
2. Select **Shared subexpression**: both comparisons point to the same `x + y` node.
3. Select **Simplification**, then toggle **Show optimized graph** to remove redundant work.
4. Delete a closing parenthesis: the message changes while the last valid graph stays visible.
5. Change a variable width in the right-hand table and watch its type label update.
6. Add a variable, rename it `budget`, set its width to 5, then type `budget >= 12`. Try a duplicate name to see validation.

This is the qlang semantic expression graph, not Python's syntax AST. No basis states are enumerated, and arithmetic being visible here does not imply the circuit compiler supports every operation. The elapsed time measures Python parsing and SVG construction, not browser paint or frontend/kernel communication.

Implementation references: [ipywidgets events](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20Events.html) · [pip GitHub installs](https://pip.pypa.io/en/stable/topics/vcs-support/).
